In [1]:
import matplotlib.pyplot as plt
import pickle
import numpy as np
from tqdm.notebook import tqdm
from hmmlearn import hmm, vhmm

from synthetic_data_generation_functions import *
from synthetic_data_analysis_functions import *
from hmm_functions import *

plt.style.use('../../paper.mplstyle')

print(plt.get_backend())
%matplotlib qt5
print(plt.get_backend())

module://matplotlib_inline.backend_inline
qt5agg


QSocketNotifier: Can only be used with threads started with QThread


# Simulations generations

## Parameters

In [ ]:
p_cw_reward = 0.8
p_ccw_reward = 0
p_cw_init = 0.5
steps_number = 100
noise_amplitude = 0.1
drift_matrix = np.array([[ 0.05 , -0.05],
                         [ 0    , 0   ]])
drift_init = 0.0

args_dict = {'p_cw_reward': p_cw_reward, 'p_ccw_reward': p_ccw_reward, 'p_cw_init': p_cw_init, 'steps_number': steps_number, 'noise_amplitude': noise_amplitude, 'drift_matrix': drift_matrix, 'drift_init': drift_init}

number_of_simulations_perset = 20

n_simulations_list = [number_of_simulations_perset]*8

start_index = 22

# fit_type = 'aic'
fit_type = 'score'

simulations_folder_path = f'/home/david/Documents/code/DDM_v5_synthetic_data_identical_drifts_{fit_type}'

## Generation

In [23]:

generate_simulations = False # PARAMS

for i, n_simulations in enumerate(n_simulations_list):
    
    if not(generate_simulations):

        break

    simulations_batch = run_simulations_batch(args_dict, n_simulations)

    with open(f'{simulations_folder_path}/n_{n_simulations}/simulations_batch_{n_simulations}_test_{start_index + i+1}.pkl', 'wb') as file:
        pickle.dump(simulations_batch, file)

# HMM fit

## Parameters

In [ ]:
n_to_test = np.arange(2,30)


## Model fit

In [25]:
fit_model = True # PARAM

if fit_model:

    for index, n_simulations in enumerate(tqdm(n_simulations_list)):
    # for index, n_simulations in enumerate(n_simulations_list):

        ####################
        ### Loading Data ###
        ####################

        with open(f'{simulations_folder_path}/n_{n_simulations}/simulations_batch_{n_simulations}_test_{start_index + index+1}.pkl', 'rb') as file:
            synthetic_data = pickle.load(file)

        ########################
        ### Reformating Data ###
        ########################

        slice_size = int(n_simulations/2)

        # synthetic_data = synthetic_data[:int(slice_size/2)] + synthetic_data[int(slice_size):int(3*slice_size/2)] + synthetic_data[int(slice_size/2):int(slice_size)] + synthetic_data[int(3*slice_size/2):]

        training_qts_sequences = [synthetic_data[i]['choices'] for i in np.arange(0,slice_size)]
        validation_qts_sequences = [synthetic_data[i]['choices'] for i in np.arange(slice_size,2*slice_size)]
        training_rewards_sequences = [synthetic_data[i]['rewards'] for i in np.arange(0,slice_size)]
        validation_rewards_sequences = [synthetic_data[i]['rewards'] for i in np.arange(slice_size,2*slice_size)]

        # action_types = np.array([[0, 1], # CCW Unrewarded, CCW Rewarded
        #                          [2, 3]]) # CW Unrewarded, CW Rewarded

        action_types = np.arange(3) # CCW, U; CW, U; CW, R

        training_sequences = []

        for training_qts_sequence, training_rewards_sequence in zip(training_qts_sequences, training_rewards_sequences):

            training_sequences.append([])

            for qt, reward in zip(training_qts_sequence, training_rewards_sequence):

                training_sequences[-1].append(action_types[qt + reward])


        validation_sequences = []

        for validation_qts_sequence, validation_rewards_sequence in zip(validation_qts_sequences, validation_rewards_sequences):

            validation_sequences.append([])

            for qt, reward in zip(validation_qts_sequence, validation_rewards_sequence):

                validation_sequences[-1].append(action_types[qt + reward])

        ###################
        ### Infer model ###
        ###################

        ## Reformating data

        reformated_training_sequences = np.array([]).astype(int)
        reformated_validation_sequences = np.array([]).astype(int)

        for x,y in zip(training_sequences,validation_sequences):

            reformated_training_sequences = np.concatenate((reformated_training_sequences, x))
            reformated_validation_sequences = np.concatenate((reformated_validation_sequences, y))

        reformated_training_sequences = reformated_training_sequences.reshape(-1,1)
        reformated_training_sequences_lengths = [len(x) for x in training_sequences]

        reformated_validation_sequences = reformated_validation_sequences.reshape(-1,1)
        reformated_validation_sequences_lengths = [len(y) for y in validation_sequences]

        ###################
        ### Infer model ###
        ###################

        if fit_type == 'score':

            best_model, best_score = infer_best_model_score(reformated_training_sequences, reformated_validation_sequences, 
                                                    reformated_training_sequences_lengths, reformated_validation_sequences_lengths, 
                                                    n_to_test, n_fits = 200, leave_loading_bar=False, verbose=False)
            
            ##################
            ### Save model ###
            ##################
            
            with open(f'{simulations_folder_path}/n_{n_simulations}/best_model_score_{n_simulations}_test_{start_index + index+1}.pkl', 'wb') as file:
                pickle.dump(best_model, file)

        elif fit_type=='aic':

            best_model, best_score = infer_best_model_aic(reformated_training_sequences, reformated_validation_sequences, 
                                                    reformated_training_sequences_lengths, reformated_validation_sequences_lengths, 
                                                    n_to_test, n_fits = 200)
            
            ##################
            ### Save model ###
            ##################
            
            with open(f'{simulations_folder_path}/n_{n_simulations}/best_model_aic_{n_simulations}_test_{start_index + index+1}.pkl', 'wb') as file:
                pickle.dump(best_model, file)



  0%|          | 0/7 [00:00<?, ?it/s]

In [6]:
++++++++++++

SyntaxError: invalid syntax (2643378534.py, line 1)

# Drift Estimation

## Generating "theoretical" average probability sequences

In [ ]:
# generate_theoretical_sequences = False # PARAM

# direction = 'cw'

# if generate_theoretical_sequences:

#     args = synthetic_data[0]['parameters'] + [5000]
#     args[0][f'']
#     drift_range = np.linspace(0.01,0.2,200)

#     # test_average_probability_sequences = generate_test_average_probability_sequences_identical_drifts(drift_range, args)
#     test_average_probability_sequences = generate_test_average_probability_sequences_identical_drifts(drift_range, args)

#     with open(f'{simulations_folder_path}/test_average_probability_{direction}_sequences.pkl', 'wb') as file:
#         pickle.dump([drift_range, test_average_probability_sequences], file)

# else:
 
#     with open(f'{simulations_folder_path}/test_average_probability_{direction}_sequences.pkl', 'rb') as file:
#         drift_range, test_average_probability_sequences = pickle.load(file)


In [ ]:
generate_theoretical_sequences = False # PARAM

if generate_theoretical_sequences:

    with open(f'{simulations_folder_path}/n_{n_simulations}/simulations_batch_{n_simulations}_test_{start_index + 0+1}.pkl', 'rb') as file:
            synthetic_data = pickle.load(file)

    args = [synthetic_data[0]['parameters']] + [5000]
    drift_range = np.linspace(0.01,0.2,200)

    # test_average_probability_sequences = generate_test_average_probability_sequences_identical_drifts(drift_range, args)
    test_msd_sequence_list = generate_test_msd_sequences_identical_drifts(drift_range, args)

    with open(f'{simulations_folder_path}/test_msd_sequences.pkl', 'wb') as file:
        pickle.dump([drift_range, test_msd_sequence_list], file)

else:

    with open(f'{simulations_folder_path}/test_msd_sequences.pkl', 'rb') as file:
        drift_range, test_msd_sequence_list = pickle.load(file)


## Minimizing mean square error

In [ ]:
save_result = True # PARAM
average_proba_sequences_hmm = []
msd_sequences_hmm = []
nbr_of_states = []
ordered_action_matrixes = []

fit_type = 'score' # 'score'

# for index, _ in enumerate(tqdm(n_simulations_list)):
for index, _ in enumerate(n_simulations_list):

    ##############################
    ### Loading Data and Model ###
    ##############################

    with open(f'{simulations_folder_path}/n_{n_simulations}/simulations_batch_{n_simulations}_test_{start_index + index+1}.pkl', 'rb') as file:
        synthetic_data = pickle.load(file)

    with open(f'{simulations_folder_path}/n_{n_simulations}/best_model_score_{n_simulations}_test_{start_index + index+1}.pkl', 'rb') as file:
        model = pickle.load(file)

    nbr_of_states.append(len(model.transmat_))

    ########################
    ### Reformating Data ###
    ########################

    test_data = [synth_data['choices'] for synth_data in synthetic_data]

    initial_state_list = []
    sequences_number = len(test_data)

    for i in range(sequences_number):
        
        choices_sequence = test_data[i]
        
        states_sequence = model.predict(np.int16(choices_sequence.reshape(-1,1)))
        initial_state_list.append(states_sequence[0])

    initial_state_list_distri = []

    for s in range(len(model.transmat_)):

        initial_state_list_distri.append(initial_state_list.count(s))

    transmat = model.transmat_
    emission_vect = model.emissionprob_[:,1]
    mat = transmat
    sorted_indexes = np.argsort(emission_vect)
    vector = np.ones([len(transmat),1])/len(transmat)

    ##

    new_transmat = order_matrix(mat, sorted_indexes)

    ##

    new_emissionmat = []
    new_initial_state_list_distri = []

    for i in sorted_indexes:
        new_emissionmat.append(model.emissionprob_[i,:])
        new_initial_state_list_distri.append(initial_state_list_distri[i])

    new_emissionmat = np.array(new_emissionmat)
    ordered_action_matrixes.append(new_emissionmat)
    new_initial_state_list_distri = np.array(new_initial_state_list_distri)/np.sum(new_initial_state_list_distri)

    ####################
    ### Computations ###
    ####################

    ## Probability sequences inference

    reconstructed_proba_sequences = infer_probability_sequence(model, test_data)

    ## MSD computation

    average_proba_sequence_hmm = []
    msd_sequence_hmm = []

    steps = np.arange(len(test_data[0]))
    new_mat_i = new_transmat

    res = (np.array(reconstructed_proba_sequences) - p_cw_init)**2 # replace np.array(test_data) with hmm-infered probability
    
    msd_sequence_hmm = np.mean(res, axis=0)

    average_proba_sequences_hmm.append(average_proba_sequence_hmm)
    msd_sequences_hmm.append(msd_sequence_hmm)


    mse_list = []

    # for test_msd_sequence in tqdm(test_msd_sequence_list, leave=False):
    for test_msd_sequence in test_msd_sequence_list:
    
        mse_list.append(compute_mean_square_error_v2(msd_sequence_hmm, test_msd_sequence))

    min_mse = np.min(mse_list)
    recovered_drift = drift_range[np.where(mse_list==min_mse)[0]]

    if not(save_result):

        continue

    ##############################
    ### Saving Recovered drift ###
    ##############################
    
    with open(f'{simulations_folder_path}/n_{n_simulations}/recovered_drift_{n_simulations}_{start_index + index+1}.pkl', 'wb') as file:
        pickle.dump([test_msd_sequence,recovered_drift], file)


In [ ]:
print(recovered_drift)

[0.06251256]


# Mean square error analysis

## Recovered drift loading

In [ ]:
recovered_drift_list = []

for index, _ in enumerate(n_simulations_list):

    with open(f'{simulations_folder_path}/n_{n_simulations}/recovered_drift_{n_simulations}_{start_index + index+1}.pkl', 'rb') as file:
        _, recovered_drift = pickle.load(file)
    
    recovered_drift_list.append(recovered_drift[0])

# Plots

In [ ]:
# fig=plt.figure(figsize=(1, 4), dpi=300, constrained_layout=False, facecolor='w')
# gs = fig.add_gridspec(1, 1)
# row = gs[:].subgridspec(1, 1, hspace=0.5)

# ax1 = plt.subplot(row[0,0])

# ax1.scatter(np.ones(len(n_simulations_list)), recovered_drift_list, marker='s', alpha=0.01, s=3, linewidth=0)

# ax1.axhline(0.05, linewidth=0.7, color='k', linestyle='--', label='Drift to recover')

# ax1.set_title(f'{len(n_simulations_list)} sets of {number_of_simulations_perset} simulations of length {steps_number}')

# ax1.set_xticks([])
# ax1.set_ylabel('Recovered drift')

In [ ]:
fig=plt.figure(figsize=(4, 4), dpi=300, constrained_layout=False, facecolor='w')
gs = fig.add_gridspec(1, 1)
row = gs[:].subgridspec(1, 1, hspace=0.5)

ax1 = plt.subplot(row[0,0])

histo = np.histogram(recovered_drift_list, bins=np.linspace(0.01,0.2,51), density=True)
bin_width = histo[1][1] - histo[1][0]
ax1.stairs(histo[0]/np.sum(histo[0])/bin_width, histo[1], alpha=0.5, fill=True, label = 'Recovered drift probability density')

x = np.linspace(-0.2,0.2)
sigma = 0.1
ax1.plot(x, np.exp(-x**2/(2*sigma**2)) * 1/np.sqrt(2*sigma**2*np.pi), label = "Noise probability density")

ax1.axvline(0.05, linewidth=0.7, color='k', linestyle='--', label='Drift to recover')

# ax1.set_xticks([])
ax1.set_title(f'{len(n_simulations_list)} sets of {number_of_simulations_perset} simulations of length {steps_number}')

ax1.set_ylim([0,None])
ax1.set_xlabel('Recovered drift')
ax1.set_ylabel('Probability density')

ax1.legend()

NameError: name 'recovered_drift_list' is not defined

In [ ]:
fig=plt.figure(figsize=(1, 4), dpi=300, constrained_layout=False, facecolor='w')
gs = fig.add_gridspec(1, 1)
row = gs[:].subgridspec(1, 1, hspace=0.5)

ax1 = plt.subplot(row[0,0])

histo = np.histogram(recovered_drift_list, bins=np.linspace(0.01,0.2,51), density=False)
ax1.stairs(histo[0], histo[1], alpha=0.5, fill=True)
ax1.axvline(0.05, linewidth=0.7, color='k', linestyle='--', label='Drift to recover')

# ax1.set_xticks([])
ax1.set_title(f'{len(n_simulations_list)} sets of {number_of_simulations_perset} simulations of length {steps_number}')

ax1.set_ylim([0,None])
ax1.set_xlabel('Recovered drift')
ax1.set_ylabel('Number of simulations sets')

Text(0, 0.5, 'Number of simulations sets')

In [ ]:
msd_sequence_hmm_theory = []

steps = np.arange(len(test_data[0]))
new_mat_i = new_transmat

for i in steps:

    new_mat_i = np.matmul(new_mat_i,new_transmat)

    res = np.matmul(new_initial_state_list_distri,new_mat_i)*new_emissionmat[:,1]
        
    # msd = np.sum(np.matmul(new_initial_state_list_distri,new_mat_i) * (res - new_initial_state_list_distri*new_emissionmat[:,1]))**2
    msd = np.sum(np.matmul(new_initial_state_list_distri,new_mat_i) * (res - new_initial_state_list_distri*new_emissionmat[:,1])**2)
    # msd = np.sum((res - new_initial_state_list_distri*new_emissionmat[:,1])**2)
    
    msd_sequence_hmm_theory.append(msd)


In [ ]:
fig=plt.figure(figsize=(3, 3), dpi=300, constrained_layout=False, facecolor='w')
gs = fig.add_gridspec(1, 1)
row = gs[:].subgridspec(1, 1, hspace=0.5)


ax1 = plt.subplot(row[0,0])

index_simu_to_plot = 0

# ax1.plot(average_proba_sequences_hmm[index_simu_to_plot], label='Average probability infered by HMM', color='black')
ax1.plot(msd_sequences_hmm[index_simu_to_plot], label='Mean Square Displacement infered by HMM', color='black')
ax1.plot(msd_sequence_hmm_theory, color='grey')

drift_range_to_plot = [(i,drift_range[i]) for i in range(0, len(drift_range), 42)]

for i, d in drift_range_to_plot:

    ax1.plot(test_msd_sequence_list[i], label=f'Mean Square Displacement of 5000 simu. with drift={np.round(d,4)}', alpha=0.5, linestyle='--')

ax1.axhline(0.25, linewidth=0.7, color='k', linestyle='--', label='Drift to recover')

ax1.set_xticks([])
ax1.set_title(f'Set of {number_of_simulations_perset} simulations of length {steps_number}\n True drifts: {drift_matrix}, Recovered drift: {np.round(recovered_drift_list[index_simu_to_plot],4)}', fontsize=5)

ax1.set_xlabel('Step')
ax1.set_ylabel('Mean Square Displacement of P_cw')

ax1.legend()

In [ ]:
fig=plt.figure(figsize=(3, 3), dpi=300, constrained_layout=False, facecolor='w')
gs = fig.add_gridspec(1, 1)
row = gs[:].subgridspec(2, 1, hspace=0.5)

ax1 = plt.subplot(row[0,0])
ax2 = plt.subplot(row[1,0])

for i, _ in enumerate(n_simulations_list):

    ax1.plot(ordered_action_matrixes[i][:,1], label=f'', alpha=0.3, linestyle='--')
    # ax1.plot(np.linspace(0,1,len(ordered_action_matrixes[i][:,1])), ordered_action_matrixes[i][:,1], label=f'', alpha=0.3, linestyle='--')
    ax2.plot(np.diff(ordered_action_matrixes[i][:,1]), label=f'', alpha=0.3, linestyle='--')
    
ax1.set_ylabel('P_cw')
ax1.set_ylabel('P_cw difference')

ax2.set_xlabel('State')

# ax1.legend()


NameError: name 'ordered_action_matrixes' is not defined

In [ ]:
fig=plt.figure(figsize=(3, 3), dpi=300, constrained_layout=False, facecolor='w')
gs = fig.add_gridspec(1, 1)
row = gs[:].subgridspec(1, 1, hspace=0.5)

ax1 = plt.subplot(row[0,0])

for i, _ in enumerate(n_simulations_list):

    # ax1.scatter(i, np.var(np.diff(ordered_action_matrixes[i][:,1])), label=f'', alpha=0.5)
    ax1.scatter(i, np.mean(np.diff(ordered_action_matrixes[i][:,1])), label=f'', alpha=0.5)
    
    # ax1.plot(np.linspace(0,1,len(ordered_action_matrixes[i][:,1])), ordered_action_matrixes[i][:,1], label=f'', alpha=0.3, linestyle='--')
    
# ax1.set_ylabel('P_cw states difference variance')
ax1.set_xlabel('State')
ax1.set_ylabel('P_cw states average difference')


# ax1.legend()



NameError: name 'ordered_action_matrixes' is not defined

In [ ]:
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++

In [10]:
def reformat_hmm(synthetic_data, model):

    nbr_of_states = len(model.transmat_)

    ########################
    ### Reformating Data ###
    ########################

    test_data = [synth_data['choices'] for synth_data in synthetic_data]

    initial_state_list = []
    sequences_number = len(test_data)

    for i in range(sequences_number):
        
        choices_sequence = test_data[i]
        
        states_sequence = model.predict(np.int16(choices_sequence.reshape(-1,1)))
        initial_state_list.append(states_sequence[0])

    initial_state_list_distri = []

    for s in range(len(model.transmat_)):

        initial_state_list_distri.append(initial_state_list.count(s))

    transmat = model.transmat_
    emission_vect = model.emissionprob_[:,2]
    mat = transmat
    sorted_indexes = np.argsort(emission_vect)

    ##

    new_transmat = order_matrix(mat, sorted_indexes)

    ##

    new_emissionmat = []
    new_initial_state_list_distri = []

    for i in sorted_indexes:
        new_emissionmat.append(model.emissionprob_[i,:])
        new_initial_state_list_distri.append(initial_state_list_distri[i])

    new_emissionmat = np.array(new_emissionmat)
    new_initial_state_distri = np.array(new_initial_state_list_distri)/np.sum(new_initial_state_list_distri)

    return {'emissionmat':new_emissionmat, 'initial_state_distri':new_initial_state_distri, 'transmat':new_transmat}

In [ ]:
def compute_reconstructed_proba_sequence(choices_sequence, model):

    states_sequence = model.predict(np.int16(choices_sequence.reshape(-1,1)))
    # print(states_sequence)

    emissionprob = model.emissionprob_

    reconstructed_p_cw_sequence = []

    for s in states_sequence:

        reconstructed_p_cw_sequence.append(emissionprob[s][1]+emissionprob[s][2])

    return reconstructed_p_cw_sequence


batch_index = 1
simulation_index = 1

with open(f'{simulations_folder_path}/n_{n_simulations}/simulations_batch_{n_simulations}_test_{batch_index+1}.pkl', 'rb') as file:
    synthetic_data = dill.load(file)

with open(f'{simulations_folder_path}/n_{n_simulations}/best_model_{fit_type}_{n_simulations}_test_{batch_index+1}.pkl', 'rb') as file:
    model = dill.load(file)

# reformated_hmm = reformat_hmm(synthetic_data, model)

# reformated_emissionmat = reformated_hmm['emissionmat']
# reformated_initial_state_distri = reformated_hmm['initial_state_distri']
# reformated_transmat = reformated_hmm['transmat']

ddm_result = synthetic_data[simulation_index]

choices_sequence = ddm_result['choices']
p_cw_sequence = ddm_result['p_cw']
rewards_sequence = ddm_result['rewards']

# action_types = np.array([[0, 1], # CCW Unrewarded, CCW Rewarded
#                          [2, 3]]) # CW Unrewarded, CW Rewarded

action_types = np.arange(3) # CCW, U; CW, U; CW, R


action_sequence = np.array([])

for qt, reward in zip(choices_sequence, rewards_sequence):

    action_sequence = np.append(action_sequence, action_types[qt+reward])

reconstructed_proba_sequence = compute_reconstructed_proba_sequence(action_sequence, model)
steps = np.arange(len(choices_sequence))

cw_proba = model.emissionprob_[:,1] + model.emissionprob_[:,2]
# cw_proba = model.emissionprob_[:,2]

fig=plt.figure(figsize=(3, 3), dpi=300, constrained_layout=False, facecolor='w')
gs = fig.add_gridspec(1, 1)
row = gs[:].subgridspec(1, 1, hspace=0.5)

ax = plt.subplot(row[0,0])
ax.scatter(steps, p_cw_sequence, label='Proba. Sequence', color='blue', marker='+', alpha=0.5)

ax.scatter(steps, reconstructed_proba_sequence, label='Infered Proba. Sequence', color='green', marker='+', alpha=0.5)
ax.plot(steps, reconstructed_proba_sequence, color='green', marker='+', alpha=0.2)


for s in range(len(model.transmat_)):

    state_proba = cw_proba[s]

    ax.axhline(state_proba, color='k', alpha=0.3, linestyle='--', linewidth=0.5)

ax.set_title(f'From set of {number_of_simulations_perset} simulations of length {steps_number}')

ax.set_xlabel('Step')
ax.set_ylabel('P_n(CW)')
ax.set_ylim([-0.05,1.05])



(-0.05, 1.05)

In [ ]:
def compute_reconstructed_proba_sequence(choices_sequence, model):

    states_sequence = model.predict(np.int16(choices_sequence.reshape(-1,1)))
    # print(states_sequence)

    emissionprob = model.emissionprob_

    reconstructed_p_cw_sequence = []

    for s in states_sequence:

        reconstructed_p_cw_sequence.append(emissionprob[s][1]+emissionprob[s][2])

    return reconstructed_p_cw_sequence

msd_recovered_drift_lists = []

for index, _ in enumerate(n_simulations_list):

    with open(f'{simulations_folder_path}/n_{n_simulations}/simulations_batch_{n_simulations}_test_{start_index + index+1}.pkl', 'rb') as file:
        synthetic_data = pickle.load(file)

    with open(f'{simulations_folder_path}/n_{n_simulations}/best_model_score_{n_simulations}_test_{start_index + index+1}.pkl', 'rb') as file:
        model = pickle.load(file)

    test_data = [synth_data['choices'] for synth_data in synthetic_data]

    reformated_hmm = reformat_hmm(synthetic_data, model)

    reformated_emissionmat = reformated_hmm['emissionmat']
    reformated_initial_state_distri = reformated_hmm['initial_state_distri']
    reformated_transmat = reformated_hmm['transmat']

    steps = np.arange(len(test_data[0]))
    mat_i = reformated_transmat
    proba_vect = reformated_emissionmat[:,1] + reformated_emissionmat[:,2]

    hmm_theory_msd_sequence = []

    for i in steps:

        # mat_i_previous = mat_i
        mat_i = np.matmul(mat_i,reformated_transmat)
        #(np.matmul(reformated_initial_state_distri, mat_i) - np.matmul(reformated_initial_state_distri,mat_i_previous)) * reformated_emissionmat[:,1]
        
        term_1 = np.vecdot(np.vecmat(reformated_initial_state_distri, mat_i), proba_vect**2)
        term_2 = np.vecdot(reformated_initial_state_distri,proba_vect)**2
        term_3 = -2 * np.vecdot(np.vecmat(reformated_initial_state_distri, mat_i),proba_vect) * np.vecdot(reformated_initial_state_distri,proba_vect)

        msd = term_1 + term_2 + term_3

        hmm_theory_msd_sequence.append(msd)

    hmm_infered_msd_sequence = []
    stacked_reconstructed_proba_sequence = 0

    for seq in test_data:

        reconstructed_proba_sequence = compute_reconstructed_proba_sequence(seq,model)
        stacked_reconstructed_proba_sequence += (reconstructed_proba_sequence[0] - reconstructed_proba_sequence)**2

    hmm_infered_msd_sequence = stacked_reconstructed_proba_sequence/len(test_data)

    mse_list = []

    for test_msd_sequence in test_msd_sequence_list:
    
        mse_list.append(compute_mean_square_error_v2(hmm_infered_msd_sequence, test_msd_sequence))

    min_mse = np.min(mse_list)
    msd_recovered_drift = drift_range[np.where(mse_list==min_mse)[0]]
    msd_recovered_drift_lists.append(msd_recovered_drift)

fig=plt.figure(figsize=(4, 4), dpi=300, constrained_layout=False, facecolor='w')
gs = fig.add_gridspec(1, 1)
row = gs[:].subgridspec(1, 1, hspace=0.5)

ax1 = plt.subplot(row[0,0])

histo = np.histogram(msd_recovered_drift_lists, bins=np.linspace(0.01,0.2,51), density=False)
ax1.stairs(histo[0], histo[1], alpha=0.5, fill=True)
ax1.axvline(0.05, linewidth=0.7, color='k', linestyle='--', label='Drift to recover')

# ax1.set_xticks([])
ax1.set_title(f'{len(n_simulations_list)} sets of {number_of_simulations_perset} simulations of length {steps_number}')

ax1.set_ylim([0,None])
ax1.set_xlabel('Recovered drift')
ax1.set_ylabel('Number of simulations sets')


Text(0, 0.5, 'Number of simulations sets')

In [27]:
fig=plt.figure(figsize=(4, 4), dpi=300, constrained_layout=False, facecolor='w')
gs = fig.add_gridspec(1, 1)
row = gs[:].subgridspec(1, 2, hspace=0.5, width_ratios = [1,0.2])

ax1 = plt.subplot(row[0,0])
ax2 = plt.subplot(row[0,1])
batch_index = 11

for index, _ in enumerate(n_simulations_list):

    with open(f'{simulations_folder_path}/n_{n_simulations}/simulations_batch_{n_simulations}_test_{batch_index+1}.pkl', 'rb') as file:
        synthetic_data = dill.load(file)

    with open(f'{simulations_folder_path}/n_{n_simulations}/best_model_score_{n_simulations}_test_{start_index + index+1}.pkl', 'rb') as file:
        model = pickle.load(file)

    reformated_emissionmat = reformat_hmm(synthetic_data, model)['emissionmat']
    proba_dist = reformated_emissionmat[:,2]
    
    ax1.plot(proba_dist, marker='s', markersize=2)
    ax2.scatter(0,np.mean(np.diff(reformated_emissionmat[:,2])), marker='_', alpha=0.5)

ax1.set_xlabel('State')
ax1.set_ylabel('Proba to do CW,R')
ax1.set_xticks(np.arange(10))

ax2.set_ylabel('Average slope')

# ax.set_title(f'Average slope : {np.mean(np.diff(new_emissionmat[:,2]))}')


Text(0, 0.5, 'Average slope')